# Lezione 3: Simulatore Dinamico di Reti di Regolazione Genica (Hill Kinetics)
**File**: [`data/synthetic_grn.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/data/synthetic_grn.py)  
**Ruolo nel progetto**: Fornisce un benchmark biologico controllato con **Ground Truth nota**, simulando cinetiche enzimatiche e perturbazioni CRISPRi.


---
## 1. La Biologia dei Circuiti di Regolazione Genica (GRN)

Nelle cellule biologiche, i geni non agiscono in isolamento. Alcuni geni codificano per **Fattori di Trascrizione (TF)**, proteine che si legano al DNA e regolano (attivano o inibiscono) la trascrizione di altri geni bersaglio.

La concentrazione dell'RNA messaggero di un gene $i$ varia nel tempo secondo il bilancio tra **sintesi** e **degradazione**:
$$\frac{dx_i}{dt} = \underbrace{\alpha_i}_{\text{Trascrizione Basale}} + \sum_{j \in \text{Regolatori}(i)} W_{ji} \cdot \underbrace{\frac{x_j^n}{K^n + x_j^n}}_{\text{Funzione di Hill}} - \underbrace{\gamma_i x_i}_{\text{Degradazione Lineare}}$$

### Perché la Funzione di Hill?
- È il modello canonico della biochimica per la **cooperatività** dei legami molecolari.
- Per $x_j \ll K$, l'effetto è quasi nullo.
- Quando $x_j \approx K$, la trascrizione si attiva rapidamente (interruttore sigmoideo).
- Per $x_j \gg K$, la velocità di sintesi satura a 1 (capacità massima del promotore).


### 1.2 Modellazione del Knockout CRISPRi
Quando somministriamo una guida dCas9-KRAB per silenziare un gene bersaglio $k$ (CRISPR interference):
1. La sua sintesi $\alpha_k$ e la trascrizione dai suoi regolatori vengono azzerate: $\text{synthesis}_k = 0$.
2. L'espressione decade rapidamente verso zero: $\frac{dx_k}{dt} \approx -5 x_k$.
3. **Effetto a Cascata**: Tutti i geni a valle regolati da $k$ cambieranno la loro espressione in risposta!


---
## 2. Setup e Import


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.synthetic_grn import SyntheticGRNSimulator, SyntheticDataset
print('Simulatore GRN caricato!')


---
## 3. Generazione del Grafo Regolatorio Ground Truth

Istanziando `SyntheticGRNSimulator`, creiamo un grafo orientato sparso di $G$ geni con densità definita.


In [ ]:
sim = SyntheticGRNSimulator(num_genes=15, density=0.25, hill_coefficient=2.0, seed=42)
print(f'Numero di geni: {sim.num_genes}')
print(f'Numero di archi regolatori attivi (TF -> Target): {sim.num_edges}')
print(f'Forma matrice dei pesi W: {sim.W.shape}')


In [ ]:
# Visualizziamo la matrice di adiacenza W_true (Ground Truth biologica)
plt.figure(figsize=(7, 6))
sns.heatmap(sim.W.numpy(), cmap='Blues', annot=True, fmt='.1f', linewidths=0.5)
plt.title('Matrice di Connettività Regolatoria W_true (TF -> Bersaglio)', fontsize=12, fontweight='bold')
plt.xlabel('Gene Bersaglio (Target, i)', fontsize=10)
plt.ylabel('Fattore di Trascrizione (TF, j)', fontsize=10)
plt.show()


---
## 4. Simulazione dello Stato Stazionario con Rumore SDE (Langevin)

Le cellule biologiche presentano variabilità intrinseca (rumore stocastico nella trascrizione).
Simuliamo 500 cellule di controllo e 500 cellule perturbate con l'Equazione Differenziale Stocastica di Langevin:
$$dx = \text{drift}(x) \, dt + \sigma_{\text{noise}} \sqrt{dt} \, dW$$


In [ ]:
# Identifichiamo il gene 'Hub' (quello con più bersagli a valle)
out_degrees = torch.sum(sim.W > 0, dim=1).numpy()
hub_gene = int(np.argmax(out_degrees))
print(f'Gene Hub identificato: Gene {hub_gene} con {int(out_degrees[hub_gene])} geni regolati a valle!')

# Simuliamo la popolazione di controllo (sana)
ctrl_cells = sim.simulate_steady_state(num_cells=400, knockout_gene=None)
# Simuliamo la popolazione con Knockout del gene Hub
ko_cells = sim.simulate_steady_state(num_cells=400, knockout_gene=hub_gene)

print(f'Forma cellule controllo: {ctrl_cells.shape}')
print(f'Forma cellule knockout: {ko_cells.shape}')


In [ ]:
# Confronto dell'espressione genica prima e dopo il Knockout
mean_ctrl = ctrl_cells.mean(dim=0).numpy()
mean_ko = ko_cells.mean(dim=0).numpy()

plt.figure(figsize=(12, 5))
x_indices = np.arange(sim.num_genes)
width = 0.35

plt.bar(x_indices - width/2, mean_ctrl, width, label='Controllo (Sano)', color='royalblue', alpha=0.8)
plt.bar(x_indices + width/2, mean_ko, width, label=f'Knockout Gene {hub_gene}', color='crimson', alpha=0.8)

plt.title(f'Effetto Sistemico del Knockout del Gene Hub {hub_gene} su Tutti i Geni', fontsize=12, fontweight='bold')
plt.xlabel('Indice del Gene', fontsize=11)
plt.ylabel('Espressione Media (mRNA)', fontsize=11)
plt.xticks(x_indices, [f'G_{i}' for i in x_indices])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


### Analisi dei Risultati:
1. Guarda il **Gene Hub**: la sua espressione crolla quasi a zero a causa del knockout!
2. Guarda i geni a valle: poiché il gene hub fungeva da attivatore trascrizionale, anche i geni bersaglio subiscono una marcata repressione!
Questo è esattamente il fenomeno biologico non-lineare che la nostra Graph Neural Network condizionata da CFM deve imparare a prevedere.
